# First Steps

This notebook is an orientation pass for a first-time user. It answers three basic questions before you look at any internal details:

1. What truncated Neumann polynomial are we trying to evaluate?
2. Which kernels does the library already ship?
3. How do those choices change the matrix-product count at a realistic target degree?

The common target is the truncated Neumann polynomial

$$S_d(B) = I + B + B^2 + \cdots + B^{d-1}.$$ 

A radix kernel packages many powers of `B` into one outer residual-update step. The point of the library is not to hide this structure, but to make the tradeoff explicit: larger radix can reduce the asymptotic product count, but exact constructions become harder and eventually give way to certified approximate ones.

In [1]:
from pathlib import Path
import sys


def find_repo_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate repository root")


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "src"))
ROOT


PosixPath('/private/tmp/neumann-kernels')

In [2]:
from neumann_kernels import kernel_catalog, multiply_count, quinary_coeffs, radix9_coeffs

kernel_catalog()


[{'kernel_id': 'radix5',
  'display_name': 'Radix-5',
  'radix': 5,
  'num_products': 2,
  'status': 'exact',
  'update_cost': 4,
  'asymptotic_coefficient': 1.7227062322935722},
 {'kernel_id': 'radix9',
  'display_name': 'Radix-9',
  'radix': 9,
  'num_products': 3,
  'status': 'exact',
  'update_cost': 5,
  'asymptotic_coefficient': 1.5773243839286437},
 {'kernel_id': 'radix15',
  'display_name': 'Radix-15',
  'radix': 15,
  'num_products': 4,
  'status': 'certified approximate',
  'update_cost': 6,
  'asymptotic_coefficient': 1.5357481488588929},
 {'kernel_id': 'radix24',
  'display_name': 'Radix-24',
  'radix': 24,
  'num_products': 5,
  'status': 'certified approximate',
  'update_cost': 7,
  'asymptotic_coefficient': 1.526730043898721}]

The catalog is the library's main entry point.

- Radix-5 and radix-9 are exact kernels bundled directly as coefficient helpers.
- Radix-15 and radix-24 are shipped as certified approximate kernels with machine-readable certificate data.

In practice, the first user decision is usually not "which formula should I derive," but "which kernel should I try for my target degree and error budget?"

The exact helpers expose the coefficient vectors for the smallest exact kernels in the package. Inspecting them is useful because it makes the object concrete before moving on to higher-radix certified data.

In [3]:
quinary = quinary_coeffs()
radix9 = radix9_coeffs()
print("Nonzero quinary coefficients:", quinary[:5])
print("Nonzero radix-9 coefficients:", radix9[:9])


Nonzero quinary coefficients: [1. 1. 1. 1. 1.]
Nonzero radix-9 coefficients: [1. 1. 1. 1. 1. 1. 1. 1. 1.]


Now close the loop with the actual selection question. Suppose you want a truncated degree of `4096`. The next cell compares the matrix-product count implied by each shipped kernel family.

In [4]:
target_degree = 4096
for row in kernel_catalog():
    count = multiply_count(row["radix"], row["num_products"], target_degree)
    print(f"{row['display_name']:<9} -> {count} matrix products for degree {target_degree}")
print(f"Direct evaluation -> {target_degree - 1} matrix products")


Radix-5   -> 24 matrix products for degree 4096
Radix-9   -> 20 matrix products for degree 4096
Radix-15  -> 24 matrix products for degree 4096
Radix-24  -> 21 matrix products for degree 4096
Direct evaluation -> 4095 matrix products


This notebook answers the first orientation question: which kernel family is even worth considering. The next notebook turns to the certified approximate kernels and explains how the shipped certificate data should be read.